# Notebook 9 – Grouping & Aggregation


## Sample DataFrame

The following sample data is created in memory, so no external files are needed. Run this cell first, then run each topic in order.


In [1]:
import pandas as pd

# Sample dataset used for all topics
df = pd.DataFrame({
    "Name": ["Asha", "Ravi", "Kiran", "Priya", "Arun", "Meena", "John", "Neha"],
    "City": ["Bengaluru", "Mysuru", "Hassan", "Bengaluru", "Hassan", "Mysuru", "Bengaluru", "Mysuru"],
    "Course": ["Python", "Python", "ML", "ML", "Python", "ML", "Python", "ML"],
    "Marks": [82, 76, 91, 88, 69, 95, 79, 86],
    "Result": ["Pass", "Pass", "Pass", "Pass", "Fail", "Pass", "Pass", "Pass"]
})
print(df)

    Name       City  Course  Marks Result
0   Asha  Bengaluru  Python     82   Pass
1   Ravi     Mysuru  Python     76   Pass
2  Kiran     Hassan      ML     91   Pass
3  Priya  Bengaluru      ML     88   Pass
4   Arun     Hassan  Python     69   Fail
5  Meena     Mysuru      ML     95   Pass
6   John  Bengaluru  Python     79   Pass
7   Neha     Mysuru      ML     86   Pass


## 1. groupby()

### Definition
groupby() divides DataFrame records into groups based on one or more columns. Each group contains rows sharing the same category. We can then calculate statistics for each group separately.

### Real-Life Example
A company groups sales records by branch to compare branch performance. Each branch has many transactions. Grouping makes it possible to calculate total sales for every branch.

### AI/ML Example
In AI/ML, groupby() helps examine how features differ between categories. We can compare average marks across cities or mean values for target classes. These summaries support exploratory analysis and feature engineering.

### Explanation
df.groupby("City") forms groups for each unique city. Selecting ["Marks"] chooses the numerical column to summarize. mean() calculates the average marks in each group.

### Python Implementation


In [3]:
print(df.groupby("City")["Marks"].mean())

City
Bengaluru    83.000000
Hassan       80.000000
Mysuru       85.666667
Name: Marks, dtype: float64


## 2. agg()

### Definition
agg() applies one or more aggregation functions to grouped or ungrouped data. Common functions include sum, mean, min, max, and count. It is useful for creating several summary statistics together.

### Real-Life Example
A manager wants total, average, and maximum sales for each branch. Rather than calculating them separately, agg() produces all the required statistics in one result. This makes reporting simpler.

### AI/ML Example
In AI/ML, agg() summarizes feature distributions for different groups. For example, we can calculate mean, minimum, and maximum marks by city. Such summaries help detect differences across categories.

### Explanation
First, groupby() separates the records by city. Then agg(["mean", "min", "max"]) applies three functions to the Marks column. The result is a compact table of group-level statistics.

### Python Implementation


In [5]:
print(df.groupby("City")["Marks"].agg(["mean", "min", "max"]))

                mean  min  max
City                          
Bengaluru  83.000000   79   88
Hassan     80.000000   69   91
Mysuru     85.666667   76   95


## 3. transform()

### Definition
transform() calculates a group-based result while keeping the original number of rows. The output aligns with the original DataFrame index. It is useful when each record needs a statistic from its own group.

### Real-Life Example
A company wants every employee record to show the average salary of that employee’s department. transform() calculates the department average and repeats it for each employee in that department. The original employee rows remain available.

### AI/ML Example
In AI/ML, transform() supports group-wise feature engineering. For example, each student can receive the average marks for their city as an additional feature. The resulting feature can be used in further analysis or preprocessing.

### Explanation
groupby("City")["Marks"].transform("mean") computes each city’s mean and maps it back to every original row. Unlike agg(), it does not collapse each group to one row. The output can be assigned directly as a new column.

### Python Implementation


In [7]:
result = df.copy()
result["City_Avg_Marks"] = result.groupby("City")["Marks"].transform("mean")
print(result)

    Name       City  Course  Marks Result  City_Avg_Marks
0   Asha  Bengaluru  Python     82   Pass       83.000000
1   Ravi     Mysuru  Python     76   Pass       85.666667
2  Kiran     Hassan      ML     91   Pass       80.000000
3  Priya  Bengaluru      ML     88   Pass       83.000000
4   Arun     Hassan  Python     69   Fail       80.000000
5  Meena     Mysuru      ML     95   Pass       85.666667
6   John  Bengaluru  Python     79   Pass       83.000000
7   Neha     Mysuru      ML     86   Pass       85.666667


## 4. filter()

### Definition
GroupBy filter() keeps or removes entire groups according to a condition. The condition is evaluated separately for each group. It is different from ordinary row filtering because the decision applies to all rows in a group.

### Real-Life Example
A company may keep only branches whose average monthly sales exceed a target. Every record from qualifying branches is retained. Records from other branches are excluded.

### AI/ML Example
In AI/ML, group filtering can remove categories with too few samples or groups failing quality criteria. For example, we can keep only cities whose average marks exceed a threshold. This helps build a dataset focused on qualifying groups.

### Explanation
The lambda function receives one group at a time. It calculates the group’s average Marks and checks whether that average is greater than 80. All rows from groups satisfying the condition are returned.

### Python Implementation


In [9]:
result = df.groupby("City").filter(lambda group: group["Marks"].mean() > 80)
print(result)

    Name       City  Course  Marks Result
0   Asha  Bengaluru  Python     82   Pass
1   Ravi     Mysuru  Python     76   Pass
3  Priya  Bengaluru      ML     88   Pass
5  Meena     Mysuru      ML     95   Pass
6   John  Bengaluru  Python     79   Pass
7   Neha     Mysuru      ML     86   Pass


## 5. Pivot Tables

### Definition
A pivot table summarizes data by arranging categories into rows and columns. Pandas provides pivot_table() to calculate values such as averages, totals, or counts. It is useful for comparing several categories at once.

### Real-Life Example
A business can summarize sales by branch and product type in a single table. Branches appear as rows and products as columns. This makes comparisons across different combinations easier.

### AI/ML Example
In AI/ML, pivot tables help explore relationships between categorical features and numerical outcomes. For example, average marks can be compared across city and course combinations. Such summaries reveal patterns before model training.

### Explanation
pd.pivot_table() uses index for row groups, columns for column groups, and values for the numerical data. aggfunc determines the summary calculation. In this example, the table shows average marks by City and Course.

### Python Implementation


In [11]:
pivot = pd.pivot_table(df, values="Marks", index="City", columns="Course", aggfunc="mean")
print(pivot)

Course       ML  Python
City                   
Bengaluru  88.0    80.5
Hassan     91.0    69.0
Mysuru     90.5    76.0


## 6. Crosstab

### Definition
A crosstab is a frequency table that counts combinations of categories. Pandas provides pd.crosstab() to compare two or more categorical variables. It is useful for understanding how categories are distributed together.

### Real-Life Example
A company can count customers by city and membership type. Each cell shows how many customers belong to that combination. This helps identify the most common customer groups.

### AI/ML Example
In AI/ML, crosstab helps examine relationships between categorical features and target classes. For example, we can count Pass and Fail results in each city. These counts can reveal class imbalance or group differences.

### Explanation
pd.crosstab() counts how often combinations of categories occur. The first argument becomes the row categories and the second becomes the column categories. It produces counts without needing a numerical values column.

### Python Implementation


In [13]:
result = pd.crosstab(df["City"], df["Result"])
print(result)

Result     Fail  Pass
City                 
Bengaluru     0     3
Hassan        1     1
Mysuru        0     3
